# 第 9 章 · Prompt、YAML 配置与 Jinja 模板

**这一章你会得到什么**：搞清楚 Agent 的“性格”是怎么被 YAML + Jinja 模板注入的——改行为不用改代码，只改配置。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/agents/default.py` **L19–35** — `AgentConfig`（哪些字段能被 YAML 覆盖）
- `src/minisweagent/agents/default.py` **L52–67** — `get_template_vars` / `_render_template`（模板变量来源）
- `src/minisweagent/config/default.yaml` **L1–171** — 三段配置 + system/instance/observation/format_error 模板

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：配置的三段式

`default.yaml` 分 `agent` / `environment` / `model` 三段，分别喂给三层的 config。
`system_template` 和 `instance_template` 是 Jinja 模板，`run()` 时用 `get_template_vars()` 渲染。

## 实验 1：加载内置 default.yaml，看三段结构

In [ ]:
import yaml
cfg = yaml.safe_load((SRC / "minisweagent/config/default.yaml").read_text())
print("顶层段:", list(cfg))
print("agent 段字段:", list(cfg["agent"]))
print("model 段字段:", list(cfg["model"]))
print("environment.env:", cfg["environment"]["env"])

## 实验 2：亲手渲染 instance_template

模板里有 `{{task}}` 和 `{{system}}`/`{{release}}` 等系统变量。喂进变量渲染看看。

In [ ]:
from jinja2 import Template
import platform
tmpl = Template(cfg["agent"]["instance_template"])
rendered = tmpl.render(task="修复登录失败的 bug", **platform.uname()._asdict())
print(rendered[:400])

## 观察点
- 模板里那句 `{%- if system == "Darwin" -%}` 会根据操作系统给出不同的 `sed -i` 用法——**同一份配置，在 Mac 和 Linux 上渲染出不同 prompt**。这是“配置即行为”的典型。
- `AgentConfig`（下面这格）定义了哪些字段能被 YAML 覆盖：step_limit / cost_limit / 各种 limit。

In [ ]:
show_source("src/minisweagent/agents/default.py", 19, 35)

## 实验 3：改配置就能改行为——用不同 cost_limit 造两个 Agent

不改一行 Agent 代码，只把配置里的 `cost_limit` 换掉，行为就变了。

In [ ]:
from minisweagent.agents.default import DefaultAgent, AgentConfig
a = AgentConfig(system_template="s", instance_template="{{task}}", cost_limit=3.0)
b = AgentConfig(system_template="s", instance_template="{{task}}", cost_limit=0.0, step_limit=10)
print("A: cost_limit=", a.cost_limit, "step_limit=", a.step_limit)
print("B: cost_limit=", b.cost_limit, "step_limit=", b.step_limit)

## 动手：给 instance_template 加一条你自己的规则

取出 `cfg["agent"]["instance_template"]`，在末尾拼一句你的自定义要求（例如“完成后请输出改动摘要”），
再用 `Template(...).render(task=...)` 渲染，确认你的话出现在结果里。

In [ ]:
my_template = cfg["agent"]["instance_template"] + "\n\n额外要求：完成后请用一句话总结你改了什么。"
# TODO: 用 Template(my_template).render(task="...", **platform.uname()._asdict()) 渲染并 print 最后 200 字
...

## 闭卷检查
1. 配置分哪三段？各喂给谁？
2. system/instance 模板在什么时候被渲染？变量从哪来？
3. 举一个“改配置即改行为”的例子。